# Interactive Machine Learning Algorithm Explorer

**Start from the best use case**, then explore algorithm choices, assumptions, code, output data, and a live visualization. This notebook is **standalone**: it does not import any companion project files.

Run all cells once, then use the dropdown and buttons in the last cell. Synthetic data is for education, not model benchmarking. Requires `numpy pandas matplotlib scikit-learn torch ipywidgets`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


## Algorithm catalog and assumptions

In [ ]:
"""Educational use-case catalog. Expands named LinkedIn algorithms with common counterparts."""
ALGORITHMS = [
('Predict a numeric value','Linear Regression','supervised','Numeric target; approximately additive/linear conditional mean; observations independent when using standard inference; severe collinearity destabilizes coefficients; residual patterns, leverage and heteroscedasticity merit checks.'),
('Predict a yes/no outcome','Logistic Regression','supervised','Binary labels; log-odds approximately linear in inputs; independent observations for standard inference; scaled features and regularization help; probabilities may require calibration.'),
('Predict a numeric value robustly','Random Forest Regressor','supervised','Independent representative rows; enough samples per leaf; cannot reliably extrapolate beyond training targets; temporal leakage and high serving costs need attention.'),
('Classify tabular records','Random Forest Classifier','supervised','Labeled representative examples; correlated predictors accepted; individual predictions less interpretable; imbalance must be handled; large ensembles can be slow.'),
('Predict complex tabular patterns','Gradient Boosting','supervised','Reliable labels and validation split; sensitive to tuning, noise and leakage; early stopping/regularization reduces overfit; tabular structured data is typical.'),
('Classify with a maximum margin','Support Vector Machine','supervised','Margin-based classification; features generally scaled; kernel and C tune complexity; large datasets can strain training; scores not calibrated probabilities.'),
('Classify from nearest neighbors','K-Nearest Neighbors','supervised','Distances encode meaningful similarity; features scaled; susceptible to high-dimensional concentration, irrelevant features, and expensive inference.'),
('Model text with word frequencies','Naive Bayes','supervised','Conditional feature independence given class is a simplifying approximation; nonnegative counts for multinomial version; poor calibrated probabilities possible.'),
('Train a small dense neural model','Neural Network (MLP)','supervised','Numeric normalized features; representative labels; sufficiently large training set; nonlinear optimization and tuning; track overfitting and seed variability.'),
('Recognize image-like patterns','Convolutional Neural Network','supervised','Spatially local and repeated patterns; tensors with channel/spatial dimensions; sufficient images or augmentation; distribution shifts can hurt.'),
('Predict sequential signals','Recurrent Neural Network','supervised','Ordered sequence input; temporal signal exists; sequence windows cannot leak future values; plain RNN can struggle with long dependencies.'),
('Learn from token sequences','Transformer Encoder','supervised','Token or time sequence encoded with position information; attention compute rises with sequence length; smaller datasets may not justify complexity.'),
('Group spherical clusters','K-Means','unsupervised','Euclidean distance meaningful after scaling; roughly compact convex clusters; specify k; sensitive to outliers, initializations and uneven density.'),
('Explore nested clusters','Hierarchical Clustering','unsupervised','Chosen distance and linkage represent desired grouping; scaling matters; large N can cost substantial memory and compute.'),
('Find irregular dense clusters','DBSCAN','unsupervised','Density-connected groups separated by sparse areas; eps/min_samples tuned in scaled feature space; varying densities and high dimensions cause trouble.'),
('Detect anomalies through reconstruction','Autoencoder','unsupervised','Train mostly on normal representative data; anomalies reconstruct less well only when learnable; reconstruction error threshold calibrated on validation data.'),
('Reduce correlated features','Principal Component Analysis','reduction','Numeric continuous features; standardize when units differ; linear projections maximize variance, not necessarily task signal; low-variance predictive directions can disappear.'),
]


## Executable Python demonstrations (edit these functions to experiment)

In [ ]:
"""One runnable minimal learning experiment per algorithm, with a visualizable result."""
import numpy as np
from sklearn.datasets import make_regression, make_classification, make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans,AgglomerativeClustering,DBSCAN
from sklearn.decomposition import PCA
import torch
from torch import nn

def run_algorithm(name, seed=42):
    """Returns (dataframe, metric string, chart title) for the selected algorithm."""
    import pandas as pd
    np.random.seed(seed); torch.manual_seed(seed)
    reg=name in ('Linear Regression','Random Forest Regressor')
    if reg:
        X,y=make_regression(n_samples=240,n_features=3,noise=19,random_state=seed)
        model=LinearRegression() if name=='Linear Regression' else RandomForestRegressor(n_estimators=60,random_state=seed)
        a,b=train_test_split(np.arange(len(y)),test_size=.25,random_state=seed)
        model.fit(X[a],y[a]); pred=model.predict(X[b]); score=f'Test RMSE: {np.sqrt(mean_squared_error(y[b],pred)):.3f}'
        return pd.DataFrame({'x':y[b], 'y':pred,'split':'test'}),score,'Actual vs predicted (ideal diagonal)'
    if name in ('K-Means','Hierarchical Clustering','DBSCAN','Principal Component Analysis','Autoencoder'):
        if name=='DBSCAN': X,_=make_moons(n_samples=260,noise=.07,random_state=seed)
        else: X,_=make_blobs(n_samples=260,centers=3,cluster_std=1.2,random_state=seed)
        X=StandardScaler().fit_transform(X)
        if name=='K-Means': labels=KMeans(n_clusters=3,n_init=10,random_state=seed).fit_predict(X)
        elif name=='Hierarchical Clustering': labels=AgglomerativeClustering(n_clusters=3).fit_predict(X)
        elif name=='DBSCAN': labels=DBSCAN(eps=.24,min_samples=5).fit_predict(X)
        elif name=='Principal Component Analysis':
            rng=np.random.default_rng(seed); X=np.column_stack([X,X[:,0]*.9+.1*rng.normal(size=len(X))]); z=PCA(n_components=2).fit_transform(X)
            return pd.DataFrame({'x':z[:,0],'y':z[:,1],'group':0}), 'Reduced 3 numeric features to 2 PCs','PCA projection'
        else:
            inp=torch.tensor(X,dtype=torch.float32); net=nn.Sequential(nn.Linear(2,4),nn.Tanh(),nn.Linear(4,1),nn.Tanh(),nn.Linear(1,4),nn.Tanh(),nn.Linear(4,2))
            opt=torch.optim.Adam(net.parameters(),lr=.02)
            for _ in range(100):
                opt.zero_grad(); pred=net(inp); loss=((pred-inp)**2).mean(); loss.backward();opt.step()
            with torch.no_grad(): recon=net(inp).numpy()
            err=np.sum((recon-X)**2,axis=1)
            return pd.DataFrame({'x':X[:,0],'y':X[:,1],'group':err}),f'Average squared reconstruction error: {np.mean(err):.4f}','Reconstruction error (continuous color)'
        groups=len(set(labels)-{-1}); metric=f'{groups} clusters found; noise label -1 when applicable'
        return pd.DataFrame({'x':X[:,0],'y':X[:,1],'group':labels}),metric,'Clusters in feature space'
    if name in ('Convolutional Neural Network','Recurrent Neural Network','Transformer Encoder'):
        n=160
        if name=='Convolutional Neural Network':
            X=torch.randn(n,1,8,8); y=(X[:,:,2:6,2:6].mean(dim=(1,2,3))>0).long()
            net=nn.Sequential(nn.Conv2d(1,4,3,padding=1),nn.ReLU(),nn.Flatten(),nn.Linear(4*8*8,2))
        else:
            X=torch.randn(n,8,3); y=(X[:,4:,0].mean(dim=1)>0).long()
            if name=='Recurrent Neural Network':
                class RNNModel(nn.Module):
                    def __init__(self): super().__init__();self.r=nn.RNN(3,12,batch_first=True);self.head=nn.Linear(12,2)
                    def forward(self,x): out,_=self.r(x);return self.head(out[:,-1,:])
                net=RNNModel()
            else:
                class TransformerModel(nn.Module):
                    def __init__(self):
                        super().__init__();self.proj=nn.Linear(3,12);self.pos=nn.Parameter(torch.zeros(1,8,12));self.attn=nn.TransformerEncoderLayer(d_model=12,nhead=3,batch_first=True,dropout=0.);self.head=nn.Linear(12,2)
                    def forward(self,x):return self.head(self.attn(self.proj(x)+self.pos)[:,4:,:].mean(dim=1))
                net=TransformerModel()
        train=torch.arange(120); test=torch.arange(120,160); opt=torch.optim.Adam(net.parameters(),lr=.02)
        for _ in range(45):
            net.train();opt.zero_grad();out=net(X[train]);loss=nn.functional.cross_entropy(out,y[train]);loss.backward();opt.step()
        net.eval()
        with torch.no_grad(): logits=net(X[test]); probs=logits.softmax(dim=1)[:,1].numpy()
        acc=accuracy_score(y[test].numpy(),probs>=.5)
        return pd.DataFrame({'x':np.arange(len(probs)),'y':probs,'group':y[test].numpy()}),f'Holdout accuracy: {acc:.2%} (toy dataset)','Predicted class-1 probabilities by sample'
    X,y=make_classification(n_samples=260,n_features=8,n_informative=5,n_redundant=1,random_state=seed)
    X=StandardScaler().fit_transform(X)
    if name=='Logistic Regression': model=LogisticRegression(max_iter=500)
    elif name=='Random Forest Classifier': model=RandomForestClassifier(n_estimators=60,random_state=seed)
    elif name=='Gradient Boosting': model=HistGradientBoostingClassifier(max_iter=75,random_state=seed)
    elif name=='Support Vector Machine': model=SVC(probability=True,random_state=seed)
    elif name=='K-Nearest Neighbors': model=KNeighborsClassifier(n_neighbors=7)
    elif name=='Naive Bayes': model=MultinomialNB();X=np.abs(X)
    elif name=='Neural Network (MLP)': model=MLPClassifier(hidden_layer_sizes=(16,),max_iter=250,random_state=seed)
    else: raise ValueError(f'Unknown algorithm: {name}')
    a,b=train_test_split(np.arange(len(y)),test_size=.25,random_state=seed,stratify=y)
    model.fit(X[a],y[a]); pred=model.predict(X[b]); probs=model.predict_proba(X[b])[:,1]
    return pd.DataFrame({'x':np.arange(len(b)),'y':probs,'group':y[b]}),f'Holdout accuracy: {accuracy_score(y[b],pred):.2%}','Predicted class-1 probabilities by sample'


## Visualization helper

In [ ]:
def plot_demo(df, title):
    fig, ax=plt.subplots(figsize=(8,5))
    if 'group' in df:
        sc=ax.scatter(df.x,df.y,c=df.group,cmap='viridis',alpha=.8);fig.colorbar(sc,ax=ax,label='Group / target / error')
    else:
        ax.scatter(df.x,df.y,alpha=.7); lo=min(df.x.min(),df.y.min());hi=max(df.x.max(),df.y.max());ax.plot([lo,hi],[lo,hi],'--',label='Perfect fit');ax.legend()
    ax.set(xlabel='X / actual / sample index',ylabel='Y / prediction',title=title);fig.tight_layout();plt.show()



## Interactive best-use-case menu

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import matplotlib.pyplot as plt
import pandas as pd

selector = widgets.Dropdown(
    options=[(f"{case}  →  {name}", name) for case,name,_,_ in ALGORITHMS],
    description="Best use case:",style={"description_width":"initial"},layout=widgets.Layout(width="95%"))
seed = widgets.IntSlider(value=42,min=1,max=1000,description="Random seed:",style={"description_width":"initial"})
assumption_btn=widgets.Button(description="Explain assumptions",button_style="info")
run_btn=widgets.Button(description="Run Python demo",button_style="success")
plot_btn=widgets.Button(description="Run + visualize")
back_btn=widgets.Button(description="Show all use cases")
output=widgets.Output()

def explain(*_):
    with output:
        clear_output(wait=True)
        case,name,family,assumptions=next(row for row in ALGORITHMS if row[1]==selector.value)
        display(Markdown(f"### {name}\n**Best use case:** {case}  \n**Category:** {family}  \n**Assumptions and checks:** {assumptions}"))
        display(Markdown("**What to verify on your data:** Separate train/test sets, prevent leakage, inspect input distributions and class balance (when relevant), benchmark against a simpler baseline, and validate under real deployment conditions."))

def demo(visual=False):
    with output:
        clear_output(wait=True)
        explain_row=next(row for row in ALGORITHMS if row[1]==selector.value)
        display(Markdown(f"### {selector.value}\n**Best use case:** {explain_row[0]}  \n**Assumptions:** {explain_row[3]}"))
        try:
            frame,metric,title=run_algorithm(selector.value,seed.value)
            display(Markdown(f"**Result:** {metric}"))
            display(frame.head(12))
            if visual: plot_demo(frame,title)
        except Exception as exc:print(type(exc).__name__,exc)

def show_all(*_):
    with output:
        clear_output(wait=True)
        display(pd.DataFrame(ALGORITHMS,columns=['Best use case','Algorithm','Family','Assumptions']))
assumption_btn.on_click(explain)
run_btn.on_click(lambda _:demo(False))
plot_btn.on_click(lambda _:demo(True))
back_btn.on_click(show_all)
display(widgets.VBox([selector,seed,widgets.HBox([assumption_btn,run_btn,plot_btn,back_btn]),output]))
explain()
